In [3]:
file_path = "data/plan_och_bygg_lag.txt"

with open(file_path, "r", encoding="utf-8") as file:
    content = file.read()

print(content[:1000])

SFS-nummer · 2010:900 · Visa register
Plan- och bygglag (2010:900)
Departement: Landsbygds- och infrastrukturdepartementet SPN BB
Utfärdad: 2010-07-01
Ändring införd: t.o.m. SFS 2025:1079
Ikraft: 2011-05-02 överg.best.

1 kap. Syfte, innehåll och definitioner

1 § I denna lag finns bestämmelser om planläggning av mark och vatten och om
byggande. Bestämmelserna syftar till att, med hänsyn till den enskilda
människans frihet, främja en samhällsutveckling med jämlika och goda sociala
levnadsförhållanden och en god och långsiktigt hållbar livsmiljö för människorna
i dagens samhälle och för kommande generationer.

2 § Det är en kommunal angelägenhet att planlägga användningen av mark och
vatten enligt denna lag.

3 § Lagen innehåller bestämmelser om 

1. lagens syfte, innehåll och definitioner (1 kap.),

2. allmänna och enskilda intressen (2 kap.),

3. översiktsplan (3 kap.),

4. reglering med detaljplan och områdesbestämmelser (4 kap.),

5. att ta fram detaljplaner och områdesbestämmelser 

In [4]:
import re
from openai import OpenAI
import os
from pathlib import Path
from dotenv import load_dotenv

In [5]:
env_path = Path.cwd().parent / ".env"
load_dotenv(dotenv_path=env_path)

True

In [7]:
class LawChunker:
    
    def __init__(self, law_path):

        with open(law_path, "r", encoding="utf-8") as file:
            self.law_text = file.read()

        self.chunks = []
        self.client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

    def chunk(self):

        paragraph_pattern = r"(?:^|\n\n)(\d+\s*[a-z]?\s*§)"
        matches = list(re.finditer(paragraph_pattern, self.law_text))

        for i, match in enumerate(matches):

            header = match.group(1).strip()
            start_pos = match.start()

            end_pos = matches[i+1].start() if i+1 < len(matches) else len(self.law_text)
            
            content = self.law_text[start_pos:end_pos].strip()
            
            chapter_context = self.law_text[:start_pos]
            chapter_pattern = r"(?:^|\n)(\d+\s*kap\.)"
            chapter_matches = re.findall(chapter_pattern, chapter_context)
            current_chapter = chapter_matches[-1] if chapter_matches else "1"

            self.chunks.append({
                "text": content,
                "metadata": {
                    "chapter": current_chapter.replace("kap.", "").strip(),
                    "paragraph": header.replace("§", "").strip(),
                    "full_reference": f"{current_chapter} {header}"
                }
            })
        
        return self.chunks
    
    def refine_chunks(self):
        # last line has no full stop -> move to next chunk
        
        refined = []
        
        for i in range(len(self.chunks)):
            current_chunk = self.chunks[i]
            
            lines = current_chunk['text'].split('\n')
            last_line = lines[-1].strip() if lines else ""

            if last_line and not last_line.endswith('.') and i + 1 < len(self.chunks):

                current_chunk['text'] = '\n'.join(lines[:-1]).strip()
                
                next_chunk = self.chunks[i+1]
                next_chunk['text'] = last_line + "\n" + next_chunk['text']
                
            refined.append(current_chunk)
        
        self.chunks = refined

        return self.chunks
    
    def embed_chunks(self, model="text-embedding-3-large"):

        if not self.chunks:
            print("No chunks to embed. Run .chunk() first.")
            return

        print(len(self.chunks), "chunks to embed.")
        texts = [c['text'].strip() for c in self.chunks if c['text'].strip()]
        print(len(texts), "non-empty chunks to embed.")
        
        print(f"Embedding {len(texts)} chunks using {model}...")
        
        try:

            response = self.client.embeddings.create(
                input=texts,
                model=model
            )
            
            for i, data in enumerate(response.data):
                self.chunks[i]['embedding'] = data.embedding
                
            print("Embedding successful!")
            
        except Exception as e:
            print(f"An error occurred during embedding: {e}")     

In [8]:
chunker = LawChunker(file_path)
law_chunks = chunker.chunk()
law_chunks = chunker.refine_chunks()

In [ ]:
law_chunks[:3]

[{'text': '1 § I denna lag finns bestämmelser om planläggning av mark och vatten och om\nbyggande. Bestämmelserna syftar till att, med hänsyn till den enskilda\nmänniskans frihet, främja en samhällsutveckling med jämlika och goda sociala\nlevnadsförhållanden och en god och långsiktigt hållbar livsmiljö för människorna\ni dagens samhälle och för kommande generationer.',
  'metadata': {'chapter': '1',
   'paragraph': '1',
   'full_reference': '1 kap. 1 §'}},
 {'text': '2 § Det är en kommunal angelägenhet att planlägga användningen av mark och\nvatten enligt denna lag.',
  'metadata': {'chapter': '1',
   'paragraph': '2',
   'full_reference': '1 kap. 2 §'}},
 {'text': '3 § Lagen innehåller bestämmelser om \n\n1. lagens syfte, innehåll och definitioner (1 kap.),\n\n2. allmänna och enskilda intressen (2 kap.),\n\n3. översiktsplan (3 kap.),\n\n4. reglering med detaljplan och områdesbestämmelser (4 kap.),\n\n5. att ta fram detaljplaner och områdesbestämmelser (5 kap.),\n\n6. genomförandet av 

In [29]:
chunker.embed_chunks(model="text-embedding-3-large")

Embedding 569 chunks using text-embedding-3-large...
Embedding successful!
